In [2]:
import pandas as pd
import numpy as np

In [8]:
df = pd.read_csv('../data/raw/HOSP_ADMIT.csv')
df.head()
TARGET_COL = 'OUTCOME'

# Get all binary columns (excluding the target) for sparsity and separation checks
binary_cols = [col for col in df.columns if df[col].nunique() == 2 and col != TARGET_COL]

In [ ]:
# 1. Prove the Imbalance of outcome category
print("--- Class Imbalance ---")
print(df['OUTCOME'].value_counts(normalize=True) * 100)

--- Class Imbalance ---
OUTCOME
0    84.058824
1    15.941176
Name: proportion, dtype: float64


In [5]:
# 2. Find High-Missingness Columns (to prove Issue 2)
print("\n--- Missing Data Percentages ---")
missing = df.isna().mean() * 100
print(missing[missing > 0].sort_values(ascending=False).head(5))


--- Missing Data Percentages ---
Series([], dtype: float64)


In [6]:
# 3. Prove the Dimensionality Trap (Zero Variance & High Correlation)
print("\n--- Dimensionality Issues ---")
# Check for columns where everyone has the exact same value (Zero Variance)
zero_var = df.columns[df.nunique() <= 1].tolist()
print(f"Zero variance columns to drop: {zero_var}")


--- Dimensionality Issues ---
Zero variance columns to drop: []


In [7]:
# Check for perfect correlation (Collinearity) in the continuous features
cont_cols = ['AGE', 'S_AD_ORIT', 'D_AD_ORIT', 'ALT_BLOOD', 'AST_BLOOD', 'L_BLOOD', 'ROE']
corr_matrix = df[cont_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [column for column in upper.columns if any(upper[column] > 0.8)]
print(f"Highly correlated continuous features: {high_corr}")

Highly correlated continuous features: ['D_AD_ORIT']


In [9]:
# ---------------------------------------------------------
# 1. The EPV (Events Per Variable) Limit
# ---------------------------------------------------------
print("\n--- 1. EPV Limit Breach ---")
minority_events = df[TARGET_COL].sum()  # Total number of deaths
safe_feature_count = minority_events // 10  # The 10-EPV rule of thumb
actual_feature_count = df.shape[1] - 1  # Total columns minus outcome

print(f"Minority Events (Deaths): {minority_events}")
print(f"Maximum Safe Features for Logistic Regression: {safe_feature_count}")
print(f"Actual Features in Dataset: {actual_feature_count}")
print(f"Conclusion: Dataset has {actual_feature_count - safe_feature_count} too many features for unregularized MLE.")


--- 1. EPV Limit Breach ---
Minority Events (Deaths): 271
Maximum Safe Features for Logistic Regression: 27
Actual Features in Dataset: 84
Conclusion: Dataset has 57 too many features for unregularized MLE.


In [10]:
# ---------------------------------------------------------
# 2. Quasi-Complete Separation
# ---------------------------------------------------------
print("\n--- 2. Quasi-Complete Separation Risks ---")
# We look for clinical variables where 100% of the patients with that condition
# either all lived or all died. This breaks standard logistic regression.
separation_warnings = []

for col in binary_cols:
    # Calculate mortality rate and count for each category (0 and 1)
    stats = df.groupby(col)[TARGET_COL].agg(['mean', 'count'])
    
    # If mean is exactly 0.0 (everyone lived) or 1.0 (everyone died), it separates the data
    perfect_sep = stats[(stats['mean'] == 0.0) | (stats['mean'] == 1.0)]
    
    # We only care if it's a positive clinical finding (e.g., they actually had the arrhythmia)
    if not perfect_sep.empty and 1 in perfect_sep.index:
        count = perfect_sep.loc[1, 'count']
        if count > 0:
            separation_warnings.append(f"Feature '{col}': {count} patients have this, and 100% of them share the same outcome.")

if separation_warnings:
    for warning in separation_warnings[:5]: # Show top 5
        print(warning)
else:
    print("No perfect mathematical separation found, but extreme sparsity is high.")


--- 2. Quasi-Complete Separation Risks ---
Feature 'nr07': 1 patients have this, and 100% of them share the same outcome.
Feature 'np07': 1 patients have this, and 100% of them share the same outcome.
Feature 'np09': 2 patients have this, and 100% of them share the same outcome.
Feature 'ritm_ecg_p_06': 1 patients have this, and 100% of them share the same outcome.
Feature 'n_r_ecg_p_09': 2 patients have this, and 100% of them share the same outcome.


outcome is 16% as dead

In [11]:
# ---------------------------------------------------------
# 3. Zero-Information (Extreme Sparsity) Features
# ---------------------------------------------------------
print("\n--- 3. Zero-Information / Extreme Sparsity Features ---")
# We look for features that occur in less than 0.5% of the patient population.
# These act purely as noise in high-dimensional unregularized models.
sparsity_threshold = 0.005 # 0.5% prevalence
sparse_cols = []

for col in binary_cols:
    prevalence = df[col].mean()
    if prevalence > 0 and prevalence < sparsity_threshold:
        sparse_cols.append((col, prevalence * 100))

print(f"Found {len(sparse_cols)} extremely sparse features (<0.5% prevalence).")
# Sort by prevalence to show the absolute rarest conditions
sparse_cols.sort(key=lambda x: x[1])
for col, prev in sparse_cols[:5]:
    print(f"Feature '{col}': Occurs in only {prev:.3f}% of patients.")


--- 3. Zero-Information / Extreme Sparsity Features ---
Found 23 extremely sparse features (<0.5% prevalence).
Feature 'nr07': Occurs in only 0.059% of patients.
Feature 'np07': Occurs in only 0.059% of patients.
Feature 'ritm_ecg_p_06': Occurs in only 0.059% of patients.
Feature 'np01': Occurs in only 0.118% of patients.
Feature 'np09': Occurs in only 0.118% of patients.


In [12]:
# 1. Calculate baseline mortality for the whole hospital cohort
baseline_mortality = df[TARGET_COL].mean() * 100
print(f"Baseline Hospital Mortality Rate: {baseline_mortality:.1f}%\n")

# 2. Analyze the extreme values of critical continuous markers
clinical_markers = ['L_BLOOD', 'ALT_BLOOD', 'AST_BLOOD']

for col in clinical_markers:
    # We define a statistical "outlier" as the top 5% of values (95th percentile)
    threshold = df[col].quantile(0.95)
    
    # Isolate the patients above this threshold
    outliers = df[df[col] > threshold]
    
    if len(outliers) > 0:
        outlier_mortality = outliers[TARGET_COL].mean() * 100
        risk_multiplier = outlier_mortality / baseline_mortality
        
        print(f"Marker: {col}")
        print(f"  95th Percentile Threshold: {threshold:.2f}")
        print(f"  Number of 'Outlier' Patients: {len(outliers)}")
        print(f"  Mortality Rate of 'Outliers': {outlier_mortality:.1f}%")
        print(f"  -> Conclusion: Truncating these outliers erases patients who are {risk_multiplier:.1f}x more likely to die.\n")

Baseline Hospital Mortality Rate: 15.9%

Marker: L_BLOOD
  95th Percentile Threshold: 15.20
  Number of 'Outlier' Patients: 85
  Mortality Rate of 'Outliers': 24.7%
  -> Conclusion: Truncating these outliers erases patients who are 1.5x more likely to die.

Marker: ALT_BLOOD
  95th Percentile Threshold: 1.20
  Number of 'Outlier' Patients: 76
  Mortality Rate of 'Outliers': 10.5%
  -> Conclusion: Truncating these outliers erases patients who are 0.7x more likely to die.

Marker: AST_BLOOD
  95th Percentile Threshold: 0.60
  Number of 'Outlier' Patients: 78
  Mortality Rate of 'Outliers': 21.8%
  -> Conclusion: Truncating these outliers erases patients who are 1.4x more likely to die.

